In [ ]:
test

In [ ]:
# ============================================================
# CLAIMS FRAUD DETECTION PIPELINE
# Motorcycle 1: Claimant-level statistical fraud detection
# Motorcycle 2: Dealer-level statistical fraud detection
#
# Purpose:
# This code creates explainable fraud risk scores using claims history.
# It does NOT require labeled fraud data.
#
# Main outputs:
# 1. claimant_scored      -> one row per claimant with risk score
# 2. dealer_scored        -> one row per dealer with risk score
# 3. review_queue         -> one row per claim, prioritized for audit review
#
# Why this approach:
# - Claims fraud is often rare and unlabeled.
# - Unsupervised anomaly detection is useful when we do not yet have confirmed
#   fraud / not-fraud labels.
# - The code combines business rules, claimant behavior patterns,
#   dealer benchmarking, fuzzy matching, and anomaly models.
# ============================================================


# ============================================================
# 1. IMPORT REQUIRED PACKAGES
# ============================================================

import pandas as pd
import numpy as np

from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer


# ============================================================
# 2. ASSUMED INPUT DATA STRUCTURE
# ============================================================

# This pipeline assumes your input dataframe is called `claims`.
#
# Expected core columns:
#
# claim_id                Unique claim identifier
# claimant_id             Unique claimant/customer/person identifier
# dealer_id               Dealer associated with the claim
# claim_date              Date claim was made or processed
# install_date            Date product/service was installed
# submission_timestamp    Exact timestamp when claim was submitted
# product_sku             Product identifier
# serial_number           Serial number or product-specific identifier
# claim_amount            Dollar amount of claim
# quantity                Quantity claimed
# claim_status            Status such as Approved, Rejected, Denied
# claimant_name           Claimant name
# claimant_email          Claimant email
# claimant_phone          Claimant phone number
# claimant_address        Claimant address
# dealer_region           Dealer region / territory / market
# program_id              Program or incentive campaign identifier
#
# If your actual column names are different, rename them before running:
#
# claims = claims.rename(columns={
#     "your_claim_column": "claim_id",
#     "your_customer_column": "claimant_id",
#     ...
# })


# ============================================================
# 3. DATA PREPARATION
# ============================================================

def prepare_claims_data(claims: pd.DataFrame) -> pd.DataFrame:
    """
    Cleans raw claims data and creates basic date/time features.

    Why this is needed:
    Fraud patterns often show up in timing behavior:
    - submissions outside business hours
    - claims submitted in bursts
    - unusual install-to-claim lag
    - period-end submission spikes

    This function standardizes dates and creates foundation fields
    used later by both claimant-level and dealer-level models.
    """

    df = claims.copy()

    # Convert date columns into pandas datetime format.
    # errors="coerce" turns invalid dates into NaT instead of crashing.
    df["claim_date"] = pd.to_datetime(df["claim_date"], errors="coerce")
    df["install_date"] = pd.to_datetime(df["install_date"], errors="coerce")
    df["submission_timestamp"] = pd.to_datetime(
        df["submission_timestamp"],
        errors="coerce"
    )

    # Extract useful calendar fields from submission timestamp.
    # These help detect timing anomalies.
    df["submission_date"] = df["submission_timestamp"].dt.date
    df["submission_hour"] = df["submission_timestamp"].dt.hour
    df["submission_dayofweek"] = df["submission_timestamp"].dt.dayofweek

    # Month is useful for dashboarding and optional sell-in comparison.
    df["submission_month"] = (
        df["submission_timestamp"]
        .dt.to_period("M")
        .astype(str)
    )

    # Flag claims submitted outside a normal business window.
    # Why:
    # Unusual after-hours or weekend submission patterns may indicate automation,
    # batching, or behavior that differs from normal claimant/dealer behavior.
    df["is_after_hours"] = (
        (df["submission_hour"] < 8) |
        (df["submission_hour"] > 18) |
        (df["submission_dayofweek"] >= 5)
    ).astype(int)

    # Compute lag between install date and claim date.
    # Why:
    # Very short, very long, or highly inconsistent lag can be suspicious.
    df["install_to_claim_days"] = (
        df["claim_date"] - df["install_date"]
    ).dt.days

    # Create rejection indicator from claim status.
    # Why:
    # High rejection rates may suggest low-quality or potentially abusive claims.
    df["is_rejected"] = (
        df["claim_status"]
        .astype(str)
        .str.lower()
        .isin(["rejected", "denied", "invalid"])
    ).astype(int)

    # Standardize text columns.
    # Why:
    # Lowercasing and trimming makes grouping, matching, and fuzzy comparison
    # more consistent.
    string_cols = [
        "claimant_id",
        "dealer_id",
        "product_sku",
        "serial_number",
        "claimant_name",
        "claimant_email",
        "claimant_phone",
        "claimant_address",
        "dealer_region",
        "program_id"
    ]

    for col in string_cols:
        if col in df.columns:
            df[col] = (
                df[col]
                .astype(str)
                .str.strip()
                .str.lower()
            )

    return df


# ============================================================
# 4. UTILITY FUNCTIONS
# ============================================================

def robust_z_score(series: pd.Series) -> pd.Series:
    """
    Calculates a robust z-score using median and MAD.

    Why use robust z-score instead of regular z-score:
    Claims and fraud data are usually skewed.
    A few extreme values can distort mean and standard deviation.
    Median and MAD are more resistant to outliers.

    Used for:
    - Dealer peer benchmarking
    - Comparing dealers within the same region
    """

    median = series.median()
    mad = np.median(np.abs(series - median))

    # If MAD is zero, all values are likely the same.
    # Return zero so we do not divide by zero.
    if mad == 0:
        return pd.Series(np.zeros(len(series)), index=series.index)

    return 0.6745 * (series - median) / mad


def entropy_from_counts(counts: pd.Series) -> float:
    """
    Calculates entropy from category counts.

    Why entropy:
    Entropy measures diversity.
    Low entropy means repetitive behavior.
    High entropy means more varied behavior.

    Fraud examples:
    - Claimant always submits the same SKU
    - Dealer claims are concentrated around very few claimants
    - Claimant always uses same dealer/product combination

    These are not automatically fraud, but they are useful risk signals.
    """

    if counts.sum() == 0:
        return 0

    probs = counts / counts.sum()

    return -np.sum(probs * np.log2(probs + 1e-9))


def gini(array) -> float:
    """
    Calculates Gini coefficient.

    Why Gini:
    Gini measures inequality or concentration.
    For dealers, it helps answer:
    Are claims evenly distributed across many claimants,
    or concentrated among only a few claimants?

    Higher Gini means higher concentration risk.
    """

    array = np.array(array, dtype=float)

    if len(array) == 0:
        return 0

    if np.amin(array) < 0:
        array = array - np.amin(array)

    array = array + 1e-9
    array = np.sort(array)

    n = len(array)
    index = np.arange(1, n + 1)

    return (
        np.sum((2 * index - n - 1) * array) /
        (n * np.sum(array))
    )


def hhi_from_counts(counts: pd.Series) -> float:
    """
    Calculates Herfindahl-Hirschman Index, or HHI.

    Why HHI:
    HHI measures concentration.
    In fraud detection, it is useful for identifying dealers where a large
    portion of claims comes from a small number of claimants.

    Higher HHI means claims are more concentrated.
    """

    if counts.sum() == 0:
        return 0

    shares = counts / counts.sum()

    return np.sum(shares ** 2)


# ============================================================
# 5. MOTORCYCLE 1:
# CLAIMANT-LEVEL FEATURE ENGINEERING
# ============================================================

def build_claimant_features(
    df: pd.DataFrame,
    audit_threshold: float = None
) -> pd.DataFrame:
    """
    Creates claimant-level behavioral features.

    Motorcycle 1 goal:
    Detect unusual claimant behavior using claims history.

    Examples of fraud signals:
    - unusually high claim volume
    - claims submitted too frequently
    - claims repeatedly just below audit threshold
    - high rejection rate
    - after-hours activity
    - low product diversity
    - highly consistent timing
    """

    work = df.copy()

    # Sort by claimant and timestamp to calculate time between submissions.
    work = work.sort_values(["claimant_id", "submission_timestamp"])

    # Previous submission timestamp for each claimant.
    work["prev_submission_timestamp"] = (
        work.groupby("claimant_id")["submission_timestamp"]
        .shift(1)
    )

    # Time since prior claim, in hours.
    # Why:
    # Very short intervals or perfectly spaced intervals can indicate batching,
    # automation, or scripted submissions.
    work["hours_since_prior_submission"] = (
        work["submission_timestamp"] -
        work["prev_submission_timestamp"]
    ).dt.total_seconds() / 3600

    # Claims just below audit threshold.
    # Why:
    # If claimants learn that claims above a certain amount are audited,
    # fraudulent behavior may cluster just below the threshold.
    if audit_threshold is not None:
        work["near_below_audit_threshold"] = (
            (work["claim_amount"] >= audit_threshold * 0.90) &
            (work["claim_amount"] < audit_threshold)
        ).astype(int)
    else:
        work["near_below_audit_threshold"] = 0

    # Daily claim counts per claimant.
    daily = (
        work.groupby(["claimant_id", "submission_date"])
        .agg(daily_claim_count=("claim_id", "count"))
        .reset_index()
    )

    daily["submission_date"] = pd.to_datetime(daily["submission_date"])
    daily = daily.sort_values(["claimant_id", "submission_date"])

    # Rolling 7-day volume.
    # Why:
    # Velocity spikes are often more useful than lifetime totals.
    # A claimant may not look suspicious overall but may suddenly spike.
    daily["rolling_7d_claim_count"] = (
        daily.groupby("claimant_id")["daily_claim_count"]
        .transform(lambda x: x.rolling(7, min_periods=1).sum())
    )

    daily_agg = (
        daily.groupby("claimant_id")
        .agg(
            max_7d_claim_count=("rolling_7d_claim_count", "max"),
            avg_daily_claim_count=("daily_claim_count", "mean"),
            std_daily_claim_count=("daily_claim_count", "std")
        )
        .reset_index()
    )

    # Product entropy per claimant.
    # Why:
    # Low product diversity may indicate repetitive exploitation of one product.
    product_entropy = (
        work.groupby("claimant_id")["product_sku"]
        .apply(lambda x: entropy_from_counts(x.value_counts()))
        .reset_index(name="product_entropy")
    )

    # Dealer entropy per claimant.
    # Why:
    # A claimant using only one dealer may be normal,
    # but combined with other high-risk signals it may matter.
    dealer_entropy = (
        work.groupby("claimant_id")["dealer_id"]
        .apply(lambda x: entropy_from_counts(x.value_counts()))
        .reset_index(name="dealer_entropy")
    )

    # Aggregate claimant behavior.
    claimant_features = (
        work.groupby("claimant_id")
        .agg(
            claim_count=("claim_id", "count"),
            unique_dealers=("dealer_id", "nunique"),
            unique_products=("product_sku", "nunique"),
            unique_programs=("program_id", "nunique"),

            total_claim_amount=("claim_amount", "sum"),
            avg_claim_amount=("claim_amount", "mean"),
            std_claim_amount=("claim_amount", "std"),

            total_quantity=("quantity", "sum"),
            avg_quantity=("quantity", "mean"),

            rejection_rate=("is_rejected", "mean"),
            after_hours_rate=("is_after_hours", "mean"),
            near_threshold_rate=("near_below_audit_threshold", "mean"),

            avg_install_to_claim_days=("install_to_claim_days", "mean"),
            std_install_to_claim_days=("install_to_claim_days", "std"),

            avg_hours_between_claims=("hours_since_prior_submission", "mean"),
            std_hours_between_claims=("hours_since_prior_submission", "std"),
            min_hours_between_claims=("hours_since_prior_submission", "min"),

            first_submission=("submission_timestamp", "min"),
            last_submission=("submission_timestamp", "max")
        )
        .reset_index()
    )

    # Number of days between first and last submission.
    # Why:
    # Helps distinguish a long-active claimant from a short burst claimant.
    claimant_features["active_days"] = (
        claimant_features["last_submission"] -
        claimant_features["first_submission"]
    ).dt.days + 1

    # Claim velocity.
    # Why:
    # Normalizes claim count by active period.
    claimant_features["claims_per_active_day"] = (
        claimant_features["claim_count"] /
        claimant_features["active_days"].replace(0, 1)
    )

    # Coefficient of variation for timing.
    # Why:
    # Very low variation can indicate mechanical or automated submissions.
    claimant_features["timing_cv"] = (
        claimant_features["std_hours_between_claims"] /
        claimant_features["avg_hours_between_claims"].replace(0, np.nan)
    )

    # Coefficient of variation for claim amount.
    # Why:
    # Identical or near-identical claim amounts repeatedly can be suspicious.
    claimant_features["amount_cv"] = (
        claimant_features["std_claim_amount"] /
        claimant_features["avg_claim_amount"].replace(0, np.nan)
    )

    # Merge all claimant feature blocks.
    claimant_features = claimant_features.merge(
        daily_agg,
        on="claimant_id",
        how="left"
    )

    claimant_features = claimant_features.merge(
        product_entropy,
        on="claimant_id",
        how="left"
    )

    claimant_features = claimant_features.merge(
        dealer_entropy,
        on="claimant_id",
        how="left"
    )

    # Fill numeric missing values.
    numeric_cols = claimant_features.select_dtypes(include=[np.number]).columns
    claimant_features[numeric_cols] = claimant_features[numeric_cols].fillna(0)

    return claimant_features


# ============================================================
# 6. MOTORCYCLE 1:
# SHARED IDENTITY FEATURES
# ============================================================

def add_shared_identity_features(
    df: pd.DataFrame,
    claimant_features: pd.DataFrame
) -> pd.DataFrame:
    """
    Adds shared identity signals.

    Why this matters:
    Fraud may be split across multiple claimant IDs that share the same:
    - email
    - phone
    - address

    This can indicate duplicate identities, household-level abuse,
    synthetic claimant creation, or coordinated behavior.
    """

    work = df.copy()
    result = claimant_features.copy()

    identity_cols = [
        "claimant_email",
        "claimant_phone",
        "claimant_address"
    ]

    for col in identity_cols:
        if col not in work.columns:
            continue

        # Count how many unique claimant IDs share the same identity attribute.
        shared_map = (
            work.groupby(col)["claimant_id"]
            .nunique()
            .reset_index(name=f"{col}_shared_claimant_count")
        )

        work = work.merge(shared_map, on=col, how="left")

        # Roll up shared identity count to claimant level.
        claimant_shared = (
            work.groupby("claimant_id")[f"{col}_shared_claimant_count"]
            .max()
            .reset_index()
        )

        result = result.merge(
            claimant_shared,
            on="claimant_id",
            how="left"
        )

    numeric_cols = result.select_dtypes(include=[np.number]).columns
    result[numeric_cols] = result[numeric_cols].fillna(0)

    return result


# ============================================================
# 7. MOTORCYCLE 1:
# FUZZY DUPLICATE DETECTION
# ============================================================

def fuzzy_duplicate_score(
    df: pd.DataFrame,
    text_col: str,
    id_col: str = "claim_id",
    min_similarity: float = 0.90
) -> pd.DataFrame:
    """
    Finds near-duplicate text values using character-level TF-IDF.

    Why this is useful:
    Exact duplicate rules catch identical values only.
    Fraudulent or duplicate claims may use small variations, such as:
    - serial number typo
    - name spelling variation
    - spacing/punctuation differences

    Why TF-IDF with character n-grams:
    Character n-grams are good for fuzzy text similarity because they detect
    partial overlaps and small edits without requiring exact matches.
    """

    work = df[[id_col, text_col]].copy()

    work[text_col] = (
        work[text_col]
        .fillna("")
        .astype(str)
        .str.lower()
        .str.strip()
    )

    nonblank = work[work[text_col] != ""].copy()

    # If fewer than two records exist, no comparison is possible.
    if len(nonblank) < 2:
        work[f"{text_col}_max_similarity"] = 0
        work[f"{text_col}_near_duplicate_flag"] = 0

        return work[
            [
                id_col,
                f"{text_col}_max_similarity",
                f"{text_col}_near_duplicate_flag"
            ]
        ]

    # Character n-gram TF-IDF.
    # analyzer="char_wb" uses character sequences inside word boundaries.
    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=1
    )

    X = vectorizer.fit_transform(nonblank[text_col])

    # Nearest neighbor finds the most similar record for each row.
    # Cosine distance is used because TF-IDF vectors are sparse text vectors.
    nn = NearestNeighbors(
        n_neighbors=min(2, len(nonblank)),
        metric="cosine"
    )

    nn.fit(X)

    distances, indices = nn.kneighbors(X)

    # First nearest neighbor is usually the record itself.
    # Second nearest neighbor is the closest other record.
    if distances.shape[1] > 1:
        nearest_distance = distances[:, 1]
    else:
        nearest_distance = distances[:, 0]

    max_similarity = 1 - nearest_distance

    nonblank[f"{text_col}_max_similarity"] = max_similarity

    nonblank[f"{text_col}_near_duplicate_flag"] = (
        nonblank[f"{text_col}_max_similarity"] >= min_similarity
    ).astype(int)

    output = work[[id_col]].merge(
        nonblank[
            [
                id_col,
                f"{text_col}_max_similarity",
                f"{text_col}_near_duplicate_flag"
            ]
        ],
        on=id_col,
        how="left"
    )

    output[f"{text_col}_max_similarity"] = (
        output[f"{text_col}_max_similarity"]
        .fillna(0)
    )

    output[f"{text_col}_near_duplicate_flag"] = (
        output[f"{text_col}_near_duplicate_flag"]
        .fillna(0)
    )

    return output


# ============================================================
# 8. MOTORCYCLE 1:
# CLAIMANT ANOMALY SCORING
# ============================================================

def score_claimant_anomalies(
    claimant_features: pd.DataFrame,
    contamination: float = 0.05
) -> pd.DataFrame:
    """
    Scores claimant-level fraud risk using Isolation Forest.

    Why Isolation Forest:
    - Works without labeled fraud data.
    - Designed to isolate unusual observations.
    - Good first model for rare anomaly detection.
    - Handles many numeric behavioral features well.

    contamination:
    Expected fraction of unusual claimants.
    Example:
    contamination=0.05 means the model expects about 5% of claimants
    to look anomalous.
    """

    result = claimant_features.copy()

    feature_cols = [
        "claim_count",
        "unique_dealers",
        "unique_products",
        "unique_programs",
        "total_claim_amount",
        "avg_claim_amount",
        "total_quantity",
        "avg_quantity",
        "rejection_rate",
        "after_hours_rate",
        "near_threshold_rate",
        "avg_install_to_claim_days",
        "std_install_to_claim_days",
        "avg_hours_between_claims",
        "std_hours_between_claims",
        "min_hours_between_claims",
        "claims_per_active_day",
        "timing_cv",
        "amount_cv",
        "max_7d_claim_count",
        "avg_daily_claim_count",
        "std_daily_claim_count",
        "product_entropy",
        "dealer_entropy"
    ]

    # Optional features that may or may not exist depending on available data.
    optional_cols = [
        "claimant_email_shared_claimant_count",
        "claimant_phone_shared_claimant_count",
        "claimant_address_shared_claimant_count",
        "max_serial_similarity",
        "serial_near_duplicate_rate",
        "max_name_similarity",
        "name_near_duplicate_rate"
    ]

    feature_cols = [
        c for c in feature_cols + optional_cols
        if c in result.columns
    ]

    # Replace infinite/missing values.
    X = (
        result[feature_cols]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    # RobustScaler is used because claims data is usually skewed.
    # It scales using median and quartiles instead of mean/std.
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X)

    model = IsolationForest(
        n_estimators=300,
        contamination=contamination,
        random_state=42
    )

    model.fit(X_scaled)

    # IsolationForest decision_function:
    # lower values mean more anomalous.
    # Multiplying by -1 makes higher values mean higher risk.
    raw_score = -model.decision_function(X_scaled)

    result["claimant_anomaly_raw_score"] = raw_score

    # Convert raw score to 0-100 for business interpretation.
    mm = MinMaxScaler(feature_range=(0, 100))

    result["claimant_anomaly_score"] = mm.fit_transform(
        raw_score.reshape(-1, 1)
    )

    # Segment risk for dashboards and reviewer queues.
    result["claimant_risk_segment"] = pd.cut(
        result["claimant_anomaly_score"],
        bins=[-1, 60, 80, 100],
        labels=["Low", "Medium", "High"]
    )

    return result


# ============================================================
# 9. MOTORCYCLE 1:
# CLAIMANT RISK DRIVER EXPLANATIONS
# ============================================================

def add_claimant_risk_drivers(
    claimant_scored: pd.DataFrame
) -> pd.DataFrame:
    """
    Adds human-readable reasons for claimant risk.

    Why this matters:
    Investigators and business users need to know WHY a claimant is risky.
    A black-box score alone is not enough for an audit workflow.

    This function ranks available driver features by percentile and returns
    the top three reason labels.
    """

    df = claimant_scored.copy()

    driver_rules = {
        "High claim velocity": "claims_per_active_day",
        "High rejection rate": "rejection_rate",
        "High after-hours activity": "after_hours_rate",
        "Claims near audit threshold": "near_threshold_rate",
        "High 7-day volume spike": "max_7d_claim_count",
        "Low product diversity": "product_entropy",
        "Low timing variation": "timing_cv",
        "Shared phone across claimants": "claimant_phone_shared_claimant_count",
        "Shared email across claimants": "claimant_email_shared_claimant_count",
        "Shared address across claimants": "claimant_address_shared_claimant_count",
        "Near-duplicate serials": "serial_near_duplicate_rate",
        "Near-duplicate names": "name_near_duplicate_rate"
    }

    percentile_cols = []

    for label, col in driver_rules.items():
        if col in df.columns:
            pct_col = f"{col}_percentile"

            # Percentile rank shows how extreme each claimant is
            # relative to other claimants.
            df[pct_col] = df[col].rank(pct=True)

            percentile_cols.append((label, pct_col))

    def top_drivers(row):
        scored = []

        for label, pct_col in percentile_cols:
            value = row[pct_col]

            if pd.notna(value):
                scored.append((label, value))

        scored = sorted(scored, key=lambda x: x[1], reverse=True)

        return "; ".join([x[0] for x in scored[:3]])

    df["top_claimant_risk_drivers"] = df.apply(top_drivers, axis=1)

    return df


# ============================================================
# 10. MOTORCYCLE 2:
# DEALER-LEVEL FEATURE ENGINEERING
# ============================================================

def build_dealer_features(
    df: pd.DataFrame,
    claimant_scored: pd.DataFrame = None
) -> pd.DataFrame:
    """
    Creates dealer-level behavioral and benchmarking features.

    Motorcycle 2 goal:
    Detect unusual dealer behavior using claims history.

    Dealer-level fraud signals:
    - dealer has unusually high claim volume
    - dealer claims are concentrated among few claimants
    - dealer has many high-risk claimants
    - dealer has temporal clustering
    - dealer has unusual rejection or after-hours patterns
    """

    work = df.copy()

    # Attach claimant risk to each claim.
    # Why:
    # Dealer risk should reflect the riskiness of claimants associated
    # with that dealer.
    if claimant_scored is not None and "claimant_anomaly_score" in claimant_scored.columns:
        work = work.merge(
            claimant_scored[
                [
                    "claimant_id",
                    "claimant_anomaly_score",
                    "claimant_risk_segment"
                ]
            ],
            on="claimant_id",
            how="left"
        )
    else:
        work["claimant_anomaly_score"] = 0
        work["claimant_risk_segment"] = "Unknown"

    # Dealer product entropy.
    # Why:
    # Low diversity may indicate repeated exploitation of the same SKU.
    dealer_product_entropy = (
        work.groupby("dealer_id")["product_sku"]
        .apply(lambda x: entropy_from_counts(x.value_counts()))
        .reset_index(name="dealer_product_entropy")
    )

    # Dealer claimant entropy.
    # Why:
    # Low claimant diversity may indicate concentration among few claimants.
    dealer_claimant_entropy = (
        work.groupby("dealer_id")["claimant_id"]
        .apply(lambda x: entropy_from_counts(x.value_counts()))
        .reset_index(name="dealer_claimant_entropy")
    )

    # Concentration metrics by dealer.
    concentration_rows = []

    for dealer_id, group in work.groupby("dealer_id"):
        claimant_counts = group["claimant_id"].value_counts()

        concentration_rows.append({
            "dealer_id": dealer_id,

            # HHI captures concentration.
            "claimant_hhi": hhi_from_counts(claimant_counts),

            # Gini captures inequality across claimants.
            "claimant_gini": gini(claimant_counts.values),

            # Top claimant share tells us how dependent the dealer is
            # on the highest-volume claimant.
            "top_claimant_share": (
                claimant_counts.max() / claimant_counts.sum()
                if claimant_counts.sum() > 0 else 0
            )
        })

    concentration = pd.DataFrame(concentration_rows)

    # Temporal clustering: number of claims submitted by dealer
    # in the same date/hour bucket.
    # Why:
    # Many claims at the same timestamp/hour can indicate batching.
    same_hour = (
        work.groupby(["dealer_id", "submission_date", "submission_hour"])
        .agg(claims_same_hour=("claim_id", "count"))
        .reset_index()
    )

    temporal = (
        same_hour.groupby("dealer_id")
        .agg(
            max_claims_same_hour=("claims_same_hour", "max"),
            avg_claims_same_hour=("claims_same_hour", "mean")
        )
        .reset_index()
    )

    # Dealer daily volume for rolling spike detection.
    dealer_daily = (
        work.groupby(["dealer_id", "submission_date"])
        .agg(daily_claim_count=("claim_id", "count"))
        .reset_index()
    )

    dealer_daily["submission_date"] = pd.to_datetime(
        dealer_daily["submission_date"]
    )

    dealer_daily = dealer_daily.sort_values(
        ["dealer_id", "submission_date"]
    )

    # Rolling 7-day claim volume per dealer.
    # Why:
    # Detects bursts and short-term spikes.
    dealer_daily["rolling_7d_claim_count"] = (
        dealer_daily.groupby("dealer_id")["daily_claim_count"]
        .transform(lambda x: x.rolling(7, min_periods=1).sum())
    )

    dealer_daily_features = (
        dealer_daily.groupby("dealer_id")
        .agg(
            max_dealer_7d_claim_count=("rolling_7d_claim_count", "max"),
            avg_dealer_daily_claim_count=("daily_claim_count", "mean"),
            std_dealer_daily_claim_count=("daily_claim_count", "std")
        )
        .reset_index()
    )

    # Core dealer-level aggregates.
    dealer_features = (
        work.groupby("dealer_id")
        .agg(
            dealer_claim_count=("claim_id", "count"),
            unique_claimants=("claimant_id", "nunique"),
            unique_products=("product_sku", "nunique"),
            unique_programs=("program_id", "nunique"),

            total_claim_amount=("claim_amount", "sum"),
            avg_claim_amount=("claim_amount", "mean"),

            total_quantity=("quantity", "sum"),
            avg_quantity=("quantity", "mean"),

            rejection_rate=("is_rejected", "mean"),
            after_hours_rate=("is_after_hours", "mean"),

            avg_install_to_claim_days=("install_to_claim_days", "mean"),
            std_install_to_claim_days=("install_to_claim_days", "std"),

            avg_claimant_risk_score=("claimant_anomaly_score", "mean"),
            max_claimant_risk_score=("claimant_anomaly_score", "max"),

            high_risk_claimant_count=(
                "claimant_risk_segment",
                lambda x: (x.astype(str) == "High").sum()
            ),

            dealer_region=("dealer_region", "first"),

            first_submission=("submission_timestamp", "min"),
            last_submission=("submission_timestamp", "max")
        )
        .reset_index()
    )

    # Dealer active period.
    dealer_features["active_days"] = (
        dealer_features["last_submission"] -
        dealer_features["first_submission"]
    ).dt.days + 1

    # Dealer claim velocity.
    dealer_features["claims_per_active_day"] = (
        dealer_features["dealer_claim_count"] /
        dealer_features["active_days"].replace(0, 1)
    )

    # Claimant-to-claim ratio.
    # Why:
    # Helps identify whether claims are spread across many claimants
    # or concentrated in a few.
    dealer_features["claimants_per_claim"] = (
        dealer_features["unique_claimants"] /
        dealer_features["dealer_claim_count"].replace(0, 1)
    )

    # Rate of high-risk claimants associated with dealer.
    dealer_features["high_risk_claimant_rate"] = (
        dealer_features["high_risk_claimant_count"] /
        dealer_features["unique_claimants"].replace(0, 1)
    )

    # Merge dealer feature blocks.
    dealer_features = dealer_features.merge(
        dealer_product_entropy,
        on="dealer_id",
        how="left"
    )

    dealer_features = dealer_features.merge(
        dealer_claimant_entropy,
        on="dealer_id",
        how="left"
    )

    dealer_features = dealer_features.merge(
        concentration,
        on="dealer_id",
        how="left"
    )

    dealer_features = dealer_features.merge(
        temporal,
        on="dealer_id",
        how="left"
    )

    dealer_features = dealer_features.merge(
        dealer_daily_features,
        on="dealer_id",
        how="left"
    )

    numeric_cols = dealer_features.select_dtypes(include=[np.number]).columns
    dealer_features[numeric_cols] = dealer_features[numeric_cols].fillna(0)

    return dealer_features


# ============================================================
# 11. MOTORCYCLE 2:
# DEALER PEER BENCHMARKING
# ============================================================

def add_dealer_peer_benchmarks(
    dealer_features: pd.DataFrame
) -> pd.DataFrame:
    """
    Compares each dealer against similar dealers in the same region.

    Why peer benchmarking:
    A dealer may look large overall but be normal for its region.
    Region-based benchmarking helps reduce false positives by comparing
    dealers to a more relevant peer group.

    This creates:
    - region median
    - ratio vs region median
    - robust z-score within region
    """

    df = dealer_features.copy()

    benchmark_cols = [
        "dealer_claim_count",
        "total_claim_amount",
        "claims_per_active_day",
        "rejection_rate",
        "after_hours_rate",
        "avg_claimant_risk_score",
        "claimant_hhi",
        "claimant_gini",
        "top_claimant_share",
        "max_dealer_7d_claim_count"
    ]

    for col in benchmark_cols:
        if col not in df.columns:
            continue

        region_median_col = f"region_median_{col}"
        ratio_col = f"{col}_vs_region_median"
        z_col = f"{col}_region_robust_z"

        # Median behavior of dealers in same region.
        df[region_median_col] = (
            df.groupby("dealer_region")[col]
            .transform("median")
        )

        # Ratio to peer median.
        df[ratio_col] = (
            df[col] /
            df[region_median_col].replace(0, np.nan)
        )

        # Robust z-score within region.
        df[z_col] = (
            df.groupby("dealer_region")[col]
            .transform(robust_z_score)
        )

    df = df.replace([np.inf, -np.inf], np.nan).fillna(0)

    return df


# ============================================================
# 12. OPTIONAL MOTORCYCLE 2:
# SELL-IN VS CLAIMS RATIO
# ============================================================

def add_sell_in_claim_ratio(
    claims_df: pd.DataFrame,
    dealer_features: pd.DataFrame,
    sell_in: pd.DataFrame
) -> pd.DataFrame:
    """
    Optional feature block if sell-in / wholesale data is available.

    Expected sell_in columns:
    - dealer_id
    - month
    - sell_in_units
    - sell_in_amount

    Why this matters:
    If claims grow much faster than dealer inventory or wholesale volume,
    it may indicate suspicious claim behavior.

    This feature is especially useful when sell-in data becomes available.
    """

    claims_monthly = claims_df.copy()

    claims_monthly["month"] = (
        claims_monthly["submission_timestamp"]
        .dt.to_period("M")
        .astype(str)
    )

    claims_monthly = (
        claims_monthly.groupby(["dealer_id", "month"])
        .agg(
            monthly_claim_count=("claim_id", "count"),
            monthly_claim_quantity=("quantity", "sum"),
            monthly_claim_amount=("claim_amount", "sum")
        )
        .reset_index()
    )

    sell = sell_in.copy()
    sell["month"] = sell["month"].astype(str)

    merged = claims_monthly.merge(
        sell,
        on=["dealer_id", "month"],
        how="left"
    )

    # Claims-to-sell-in ratio.
    # Why:
    # A very high ratio suggests claims are high relative to inventory flow.
    merged["claims_to_sell_in_units_ratio"] = (
        merged["monthly_claim_quantity"] /
        merged["sell_in_units"].replace(0, np.nan)
    )

    merged["claims_to_sell_in_amount_ratio"] = (
        merged["monthly_claim_amount"] /
        merged["sell_in_amount"].replace(0, np.nan)
    )

    dealer_sell_features = (
        merged.groupby("dealer_id")
        .agg(
            avg_claims_to_sell_in_units_ratio=(
                "claims_to_sell_in_units_ratio",
                "mean"
            ),
            max_claims_to_sell_in_units_ratio=(
                "claims_to_sell_in_units_ratio",
                "max"
            ),
            avg_claims_to_sell_in_amount_ratio=(
                "claims_to_sell_in_amount_ratio",
                "mean"
            ),
            max_claims_to_sell_in_amount_ratio=(
                "claims_to_sell_in_amount_ratio",
                "max"
            )
        )
        .reset_index()
    )

    result = dealer_features.merge(
        dealer_sell_features,
        on="dealer_id",
        how="left"
    )

    result = result.replace([np.inf, -np.inf], np.nan).fillna(0)

    return result


# ============================================================
# 13. MOTORCYCLE 2:
# DEALER ANOMALY SCORING
# ============================================================

def score_dealer_anomalies(
    dealer_features: pd.DataFrame,
    contamination: float = 0.05
) -> pd.DataFrame:
    """
    Scores dealer-level fraud risk using Isolation Forest.

    Why Isolation Forest:
    Dealer fraud patterns may be rare and unlabeled.
    Isolation Forest can detect dealers that behave differently from the
    rest of the population based on multiple features.
    """

    result = dealer_features.copy()

    feature_cols = [
        "dealer_claim_count",
        "unique_claimants",
        "unique_products",
        "unique_programs",
        "total_claim_amount",
        "avg_claim_amount",
        "total_quantity",
        "avg_quantity",
        "rejection_rate",
        "after_hours_rate",
        "avg_install_to_claim_days",
        "std_install_to_claim_days",
        "avg_claimant_risk_score",
        "max_claimant_risk_score",
        "high_risk_claimant_count",
        "high_risk_claimant_rate",
        "claims_per_active_day",
        "claimants_per_claim",
        "dealer_product_entropy",
        "dealer_claimant_entropy",
        "claimant_hhi",
        "claimant_gini",
        "top_claimant_share",
        "max_claims_same_hour",
        "avg_claims_same_hour",
        "max_dealer_7d_claim_count",
        "avg_dealer_daily_claim_count",
        "std_dealer_daily_claim_count"
    ]

    optional_cols = [
        "avg_claims_to_sell_in_units_ratio",
        "max_claims_to_sell_in_units_ratio",
        "avg_claims_to_sell_in_amount_ratio",
        "max_claims_to_sell_in_amount_ratio"
    ]

    # Add all region benchmark robust z-score columns.
    benchmark_cols = [
        c for c in result.columns
        if c.endswith("_region_robust_z")
    ]

    feature_cols = [
        c for c in feature_cols + optional_cols + benchmark_cols
        if c in result.columns
    ]

    X = (
        result[feature_cols]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X)

    model = IsolationForest(
        n_estimators=300,
        contamination=contamination,
        random_state=42
    )

    model.fit(X_scaled)

    raw_score = -model.decision_function(X_scaled)

    result["dealer_anomaly_raw_score"] = raw_score

    mm = MinMaxScaler(feature_range=(0, 100))

    result["dealer_anomaly_score"] = mm.fit_transform(
        raw_score.reshape(-1, 1)
    )

    result["dealer_risk_segment"] = pd.cut(
        result["dealer_anomaly_score"],
        bins=[-1, 60, 80, 100],
        labels=["Low", "Medium", "High"]
    )

    return result


# ============================================================
# 14. MOTORCYCLE 2:
# DEALER RISK DRIVER EXPLANATIONS
# ============================================================

def add_dealer_risk_drivers(
    dealer_scored: pd.DataFrame
) -> pd.DataFrame:
    """
    Adds human-readable dealer risk reasons.

    Why:
    Business users need to understand why a dealer was flagged.
    This supports audit review, dashboard explainability, and stakeholder trust.
    """

    df = dealer_scored.copy()

    driver_rules = {
        "High dealer claim volume": "dealer_claim_count",
        "High claims per active day": "claims_per_active_day",
        "High rejection rate": "rejection_rate",
        "High after-hours rate": "after_hours_rate",
        "High claimant concentration": "claimant_hhi",
        "High claimant inequality": "claimant_gini",
        "High top-claimant dependency": "top_claimant_share",
        "Many high-risk claimants": "high_risk_claimant_count",
        "High average claimant risk": "avg_claimant_risk_score",
        "Temporal clustering": "max_claims_same_hour",
        "High 7-day dealer spike": "max_dealer_7d_claim_count",
        "High claims-to-sell-in units ratio": "max_claims_to_sell_in_units_ratio"
    }

    percentile_cols = []

    for label, col in driver_rules.items():
        if col in df.columns:
            pct_col = f"{col}_percentile"

            df[pct_col] = df[col].rank(pct=True)

            percentile_cols.append((label, pct_col))

    def top_drivers(row):
        scored = []

        for label, pct_col in percentile_cols:
            value = row[pct_col]

            if pd.notna(value):
                scored.append((label, value))

        scored = sorted(scored, key=lambda x: x[1], reverse=True)

        return "; ".join([x[0] for x in scored[:3]])

    df["top_dealer_risk_drivers"] = df.apply(top_drivers, axis=1)

    return df


# ============================================================
# 15. FINAL REVIEW QUEUE
# ============================================================

def build_review_queue(
    claims_df: pd.DataFrame,
    claimant_scored: pd.DataFrame,
    dealer_scored: pd.DataFrame
) -> pd.DataFrame:
    """
    Creates claim-level review queue.

    Why:
    Claimant and dealer scores are useful, but audit teams usually need
    a prioritized list of individual claims to review.

    Final score combines:
    - claimant risk
    - dealer risk
    - claim-level rule score
    """

    review = claims_df.copy()

    # Attach claimant risk to each claim.
    review = review.merge(
        claimant_scored[
            [
                "claimant_id",
                "claimant_anomaly_score",
                "claimant_risk_segment",
                "top_claimant_risk_drivers"
            ]
        ],
        on="claimant_id",
        how="left"
    )

    # Attach dealer risk to each claim.
    review = review.merge(
        dealer_scored[
            [
                "dealer_id",
                "dealer_anomaly_score",
                "dealer_risk_segment",
                "top_dealer_risk_drivers"
            ]
        ],
        on="dealer_id",
        how="left"
    )

    # Claim-level rule signals.
    # Percentile rank identifies unusually high claim amounts or quantities.
    review["claim_amount_percentile"] = (
        review["claim_amount"]
        .rank(pct=True)
    )

    review["quantity_percentile"] = (
        review["quantity"]
        .rank(pct=True)
    )

    # Lightweight claim-level rule score.
    # This can be expanded later with Bike-level validation rules.
    review["claim_level_rule_score"] = (
        20 * review["is_after_hours"].fillna(0) +
        20 * review["claim_amount_percentile"].fillna(0) +
        20 * review["quantity_percentile"].fillna(0)
    )

    # Final weighted fraud risk score.
    #
    # Weighting rationale:
    # - Claimant behavior gets the highest weight because Motorcycle 1 is focused
    #   on individual claimant behavior.
    # - Dealer behavior receives strong weight because Motorcycle 2 adds systemic
    #   context.
    # - Claim-level rules provide additional immediate signals.
    #
    # These weights are configurable and can later be exposed as dashboard sliders.
    review["final_fraud_risk_score"] = (
        0.45 * review["claimant_anomaly_score"].fillna(0) +
        0.35 * review["dealer_anomaly_score"].fillna(0) +
        0.20 * review["claim_level_rule_score"].fillna(0)
    )

    review["final_risk_segment"] = pd.cut(
        review["final_fraud_risk_score"],
        bins=[-1, 60, 80, 100],
        labels=["Low", "Medium", "High"]
    )

    review = review.sort_values(
        "final_fraud_risk_score",
        ascending=False
    )

    return review


# ============================================================
# 16. END-TO-END PIPELINE FUNCTION
# ============================================================

def run_claims_fraud_pipeline(
    claims: pd.DataFrame,
    audit_threshold: float = None,
    sell_in: pd.DataFrame = None,
    claimant_contamination: float = 0.05,
    dealer_contamination: float = 0.05
):
    """
    Runs the full fraud detection pipeline.

    Inputs:
    claims:
        Raw claims history dataframe.

    audit_threshold:
        Optional dollar threshold for detecting claims just below audit limit.

    sell_in:
        Optional sell-in / wholesale dataframe.
        If not available, dealer sell-in ratio features are skipped.

    claimant_contamination:
        Expected fraction of anomalous claimants.

    dealer_contamination:
        Expected fraction of anomalous dealers.

    Outputs:
    claims_clean:
        Cleaned claim-level data.

    claimant_scored:
        Claimant-level risk scoring output.

    dealer_scored:
        Dealer-level risk scoring output.

    review_queue:
        Claim-level prioritized audit queue.
    """

    # --------------------------------------------------------
    # Step 1: Clean and enrich raw claims data
    # --------------------------------------------------------
    claims_clean = prepare_claims_data(claims)

    # --------------------------------------------------------
    # Step 2: Create claimant-level features
    # --------------------------------------------------------
    claimant_features = build_claimant_features(
        claims_clean,
        audit_threshold=audit_threshold
    )

    # --------------------------------------------------------
    # Step 3: Add shared identity risk signals
    # --------------------------------------------------------
    claimant_features = add_shared_identity_features(
        claims_clean,
        claimant_features
    )

    # --------------------------------------------------------
    # Step 4: Add fuzzy duplicate signals for serial number
    # --------------------------------------------------------
    if "serial_number" in claims_clean.columns:
        serial_fuzzy = fuzzy_duplicate_score(
            claims_clean,
            text_col="serial_number",
            min_similarity=0.92
        )
    else:
        serial_fuzzy = claims_clean[["claim_id"]].copy()

    # --------------------------------------------------------
    # Step 5: Add fuzzy duplicate signals for claimant name
    # --------------------------------------------------------
    if "claimant_name" in claims_clean.columns:
        name_fuzzy = fuzzy_duplicate_score(
            claims_clean,
            text_col="claimant_name",
            min_similarity=0.90
        )
    else:
        name_fuzzy = claims_clean[["claim_id"]].copy()

    # --------------------------------------------------------
    # Step 6: Merge fuzzy features back to claim level
    # --------------------------------------------------------
    claims_with_fuzzy = (
        claims_clean
        .merge(serial_fuzzy, on="claim_id", how="left")
        .merge(name_fuzzy, on="claim_id", how="left")
    )

    # --------------------------------------------------------
    # Step 7: Roll fuzzy duplicate signals to claimant level
    # --------------------------------------------------------
    fuzzy_agg_dict = {}

    if "serial_number_max_similarity" in claims_with_fuzzy.columns:
        fuzzy_agg_dict["max_serial_similarity"] = (
            "serial_number_max_similarity",
            "max"
        )

    if "serial_number_near_duplicate_flag" in claims_with_fuzzy.columns:
        fuzzy_agg_dict["serial_near_duplicate_rate"] = (
            "serial_number_near_duplicate_flag",
            "mean"
        )

    if "claimant_name_max_similarity" in claims_with_fuzzy.columns:
        fuzzy_agg_dict["max_name_similarity"] = (
            "claimant_name_max_similarity",
            "max"
        )

    if "claimant_name_near_duplicate_flag" in claims_with_fuzzy.columns:
        fuzzy_agg_dict["name_near_duplicate_rate"] = (
            "claimant_name_near_duplicate_flag",
            "mean"
        )

    if len(fuzzy_agg_dict) > 0:
        fuzzy_claimant_features = (
            claims_with_fuzzy.groupby("claimant_id")
            .agg(**fuzzy_agg_dict)
            .reset_index()
        )

        claimant_features = claimant_features.merge(
            fuzzy_claimant_features,
            on="claimant_id",
            how="left"
        )

    claimant_features = claimant_features.fillna(0)

    # --------------------------------------------------------
    # Step 8: Score claimant anomalies
    # --------------------------------------------------------
    claimant_scored = score_claimant_anomalies(
        claimant_features,
        contamination=claimant_contamination
    )

    # --------------------------------------------------------
    # Step 9: Add claimant risk explanations
    # --------------------------------------------------------
    claimant_scored = add_claimant_risk_drivers(claimant_scored)

    # --------------------------------------------------------
    # Step 10: Build dealer features using claimant risk context
    # --------------------------------------------------------
    dealer_features = build_dealer_features(
        claims_clean,
        claimant_scored=claimant_scored
    )

    # --------------------------------------------------------
    # Step 11: Add dealer peer benchmarks
    # --------------------------------------------------------
    dealer_features = add_dealer_peer_benchmarks(dealer_features)

    # --------------------------------------------------------
    # Step 12: Optional sell-in / wholesale comparison
    # --------------------------------------------------------
    if sell_in is not None:
        dealer_features = add_sell_in_claim_ratio(
            claims_clean,
            dealer_features,
            sell_in
        )

    # --------------------------------------------------------
    # Step 13: Score dealer anomalies
    # --------------------------------------------------------
    dealer_scored = score_dealer_anomalies(
        dealer_features,
        contamination=dealer_contamination
    )

    # --------------------------------------------------------
    # Step 14: Add dealer risk explanations
    # --------------------------------------------------------
    dealer_scored = add_dealer_risk_drivers(dealer_scored)

    # --------------------------------------------------------
    # Step 15: Create final claim review queue
    # --------------------------------------------------------
    review_queue = build_review_queue(
        claims_clean,
        claimant_scored,
        dealer_scored
    )

    return claims_clean, claimant_scored, dealer_scored, review_queue


# ============================================================
# 17. HOW TO RUN THE PIPELINE
# ============================================================

# Example:
#
# claims_clean, claimant_scored, dealer_scored, review_queue = run_claims_fraud_pipeline(
#     claims=claims,
#     audit_threshold=500,
#     sell_in=None,
#     claimant_contamination=0.05,
#     dealer_contamination=0.05
# )


# ============================================================
# 18. SAVE OUTPUTS FOR DASHBOARDING OR POWER BI
# ============================================================

# After running the pipeline, you can save the outputs:
#
# claimant_scored.to_csv("claimant_risk_scores.csv", index=False)
# dealer_scored.to_csv("dealer_risk_scores.csv", index=False)
# review_queue.to_csv("fraud_review_queue.csv", index=False)


# ============================================================
# 19. RECOMMENDED DASHBOARD VIEWS
# ============================================================

# View 1: Claimant risk table
#
# claimant_scored[
#     [
#         "claimant_id",
#         "claim_count",
#         "claimant_anomaly_score",
#         "claimant_risk_segment",
#         "top_claimant_risk_drivers"
#     ]
# ].sort_values("claimant_anomaly_score", ascending=False).head(25)


# View 2: Dealer risk table
#
# dealer_scored[
#     [
#         "dealer_id",
#         "dealer_region",
#         "dealer_claim_count",
#         "dealer_anomaly_score",
#         "dealer_risk_segment",
#         "top_dealer_risk_drivers"
#     ]
# ].sort_values("dealer_anomaly_score", ascending=False).head(25)


# View 3: Claim review queue
#
# review_queue[
#     [
#         "claim_id",
#         "claimant_id",
#         "dealer_id",
#         "claim_amount",
#         "quantity",
#         "claimant_anomaly_score",
#         "dealer_anomaly_score",
#         "claim_level_rule_score",
#         "final_fraud_risk_score",
#         "final_risk_segment",
#         "top_claimant_risk_drivers",
#         "top_dealer_risk_drivers"
#     ]
# ].head(50)